# Lab 3 · Exercise C — A real problem: OPERA with correlated systematics
Final OPERA $\\nu_\\tau$-appearance analysis (arXiv:1804.04912, Table III): 4 channels, 10 candidates.
The uncertainties on the backgrounds and on the signal have **common sources** across channels.
We encode them in a covariance matrix, **diagonalise** it, and measure the signal strength $\\mu$.

*Assumption (ours, not the collaboration's published treatment):* each source is fully correlated across the channels it touches.

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
# OPERA final analysis, Table III (arXiv:1804.04912)
chan  = ["1h", "3h", "mu", "e"]
n_obs = np.array([6, 3, 1, 0])
sig0  = np.array([2.96, 1.83, 1.15, 0.84])        # expected signal for mu = 1
bkg0  = np.array([1.43, 0.52, 0.024, 0.035])      # expected total background

# background uncertainty by source (columns) and channel (rows)
v_charm = np.array([0.03, 0.09, 0.002, 0.007])
v_had   = np.array([0.38, 0.03, 0.0,   0.0  ])
v_las   = np.array([0.0,  0.0,  0.008, 0.0  ])
REL_SIG = 0.20                                    # relative signal uncertainty, same in all channels

def cov_signal(rho):
    """Signal covariance: 20% in each channel, correlation rho between channels."""
    D = np.diag(REL_SIG * sig0)
    C = rho * np.ones((4, 4)) + (1 - rho) * np.eye(4)
    return D @ C @ D

## C1 · Covariance and diagonalisation
$\\Sigma_b=\\sum_k v_k v_k^T$ over the three sources. Implement `factor` ($A=U\\sqrt\\Lambda$, one column per non-zero eigenvalue).

In [ ]:
def factor(Sigma, tol=1e-10):
    """Return A with  A @ A.T = Sigma  and one column per NON-ZERO eigenvalue (A = U sqrt(Lambda))."""
    # TODO: w, U = np.linalg.eigh(Sigma);  keep = w > tol;  return U[:, keep] * np.sqrt(w[keep])
    return None

In [ ]:
Sigma_b = np.outer(v_charm, v_charm) + np.outer(v_had, v_had) + np.outer(v_las, v_las)
print("eigenvalues of Sigma_b:", np.linalg.eigvalsh(Sigma_b))
A_b = factor(Sigma_b)
print("independent nuisance parameters needed for the background:", A_b.shape[1])

In [ ]:
check("factor implemented", A_b is not None)
check("A A^T reproduces Sigma_b", np.allclose(A_b @ A_b.T, Sigma_b, atol=1e-10))
check("rank = number of sources (3), not number of channels (4)", A_b.shape[1] == 3)
check("a fully correlated signal needs ONE nuisance parameter", factor(cov_signal(1.0)).shape[1] == 1)

## C2 · The RooFit model
$\\beta_i=\\beta_{0i}+\\sum_k A_{ik}z_k$ and $s_i=s_{0i}+\\sum_k A^{(s)}_{ik}z^{(s)}_k$ with **independent standard-normal** $z$;
$\\lambda_i=\\mu s_i+\\beta_i$; four Poisson terms times the constraints $\\text{Gauss}(0|z_k,1)$.
Complete `build_opera`. `analyse` (given) fits, computes the 68 % interval on $\\mu$ and the significance with RooStats.

In [ ]:
def lin(name, c0, row, zlist):
    """RooFormulaVar  c0 + sum_k row[k] * z_k."""
    expr = repr(float(c0)) + "".join("+(%r)*@%d" % (float(a), k) for k, a in enumerate(row))
    al = ROOT.RooArgList()
    for z in zlist:
        al.add(z)
    return ROOT.RooFormulaVar(name, name, expr, al)

def build_opera(rho):
    Ab, As = factor(Sigma_b), factor(cov_signal(rho))
    mu = ROOT.RooRealVar("mu", "signal strength", 1, 0, 5)
    zb = [ROOT.RooRealVar("zb%d" % k, "zb", 0, -4, 4) for k in range(Ab.shape[1])]
    zs = [ROOT.RooRealVar("zs%d" % k, "zs", 0, -4, 4) for k in range(As.shape[1])]
    gb = [ROOT.RooRealVar("gb%d" % k, "gb", 0, -5, 5) for k in range(Ab.shape[1])]   # global observables
    gs = [ROOT.RooRealVar("gs%d" % k, "gs", 0, -5, 5) for k in range(As.shape[1])]
    for g in gb + gs:
        g.setConstant(True)
    one = ROOT.RooRealVar("one", "one", 1); one.setConstant(True)
    constr = [ROOT.RooGaussian("c_" + z.GetName(), "c", g, z, one) for z, g in zip(zb + zs, gb + gs)]

    ns, betas, sigs, lams, poiss = [], [], [], [], []
    for i, c in enumerate(chan):
        beta = None   # TODO: lin("beta_"+c, bkg0[i], Ab[i], zb)   expected background in channel i
        sig  = None   # TODO: lin("sig_"+c,  sig0[i], As[i], zs)   expected signal for mu = 1
        lam  = None   # TODO: RooFormulaVar "max(@0*@1+@2,1e-9)" of (mu, sig, beta)
        n_i  = ROOT.RooRealVar("n_" + c, "n", 0, 50)
        pois = ROOT.RooPoisson("pois_" + c, "pois", n_i, lam)
        ns.append(n_i); betas.append(beta); sigs.append(sig); lams.append(lam); poiss.append(pois)

    al = ROOT.RooArgList()
    for p in poiss + constr:
        al.add(p)
    pdf = ROOT.RooProdPdf("opera", "opera", al)
    obs = ROOT.RooArgSet()
    for n_i in ns:
        obs.add(n_i)
    data = ROOT.RooDataSet("data", "data", obs)
    for n_i, v in zip(ns, n_obs):
        n_i.setVal(v)
    data.add(obs)
    return dict(locals())

In [ ]:
_counter = [0]
def analyse(m):
    """Fit, 68 % interval on mu and significance of mu = 0, with RooStats."""
    _counter[0] += 1
    w = ROOT.RooWorkspace("w%d" % _counter[0])
    w.Import(m["pdf"]); w.Import(m["data"])
    nuis, glob, obs = ROOT.RooArgSet(), ROOT.RooArgSet(), ROOT.RooArgSet()
    for z in m["zb"] + m["zs"]:
        nuis.add(w.var(z.GetName()))
    for g in m["gb"] + m["gs"]:
        glob.add(w.var(g.GetName()))
    for n_i in m["ns"]:
        obs.add(w.var(n_i.GetName()))
    mc = ROOT.RooStats.ModelConfig("mc", w)
    mc.SetPdf(w.pdf("opera"))
    mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("mu")))
    mc.SetNuisanceParameters(nuis); mc.SetGlobalObservables(glob); mc.SetObservables(obs)
    data = w.data("data")
    mu = w.var("mu")
    w.pdf("opera").fitTo(data, ROOT.RooFit.GlobalObservables(glob), ROOT.RooFit.PrintLevel(-1))
    mu_hat = mu.getVal()
    plc = ROOT.RooStats.ProfileLikelihoodCalculator(data, mc)
    plc.SetConfidenceLevel(0.683)
    iv = plc.GetInterval()
    lo, hi = iv.LowerLimit(mu), iv.UpperLimit(mu)
    mu0 = ROOT.RooRealVar("mu", "mu", 0)
    plc.SetNullParameters(ROOT.RooArgSet(mu0))
    Z = plc.GetHypoTest().Significance()
    return dict(mu_hat=mu_hat, lo=lo, hi=hi, Z=Z, w=w)

In [ ]:
m0 = build_opera(0.0)                  # signal errors independent between channels
r0 = analyse(m0)
print("rho = 0: mu_hat = %.2f  [%.2f, %.2f]   Z0 = %.2f" % (r0["mu_hat"], r0["lo"], r0["hi"], r0["Z"]))

In [ ]:
check("mu_hat close to 1.1", abs(r0["mu_hat"] - 1.1) < 0.2)
check("significance of the counting model about 3.9", abs(r0["Z"] - 3.9) < 0.4)

## C3 · Scan the correlation of the signal uncertainty
The 20 % signal uncertainty comes from sources common to all channels. Compare $\\rho=0$ (independent), $0.5$, $1$ (fully correlated).

In [ ]:
table = {}
for rho in (0.0, 0.5, 1.0):
    r = analyse(build_opera(rho))
    table[rho] = r
    print("rho = %.1f: mu_hat = %.2f  [%.2f, %.2f]  width = %.2f   Z0 = %.2f" %
          (rho, r["mu_hat"], r["lo"], r["hi"], r["hi"] - r["lo"], r["Z"]))

In [ ]:
widths = {rho: r["hi"] - r["lo"] for rho, r in table.items()}
check("the interval grows with the correlation", widths[1.0] > widths[0.0])
check("the significance of mu = 0 does not depend on the signal uncertainty", abs(table[1.0]["Z"] - table[0.0]["Z"]) < 0.2)

## Questions (3–4 lines each)
1. How many independent nuisance parameters does each model need? Why is the number smaller than the number of channels?
2. By how much does the interval on $\\mu$ change between $\\rho=0$ and $\\rho=1$? Why does $Z_0$ not change?
3. Statistical or systematic: which dominates the uncertainty on $\\mu$ here? What does that say about the value of a better systematic estimate?
4. The paper quotes a larger significance than the counting model. What information have we thrown away?